# Chapter 01 — Basic Financial Analysis

> **Section 1: Corporate Finance**

## Overview

This chapter introduces the fundamental time-value-of-money concepts used throughout financial modeling. The focus is on understanding the financial mathematics behind present value, future value, investment returns, loan structures, and discounted cash flows, then implementing those concepts computationally using Python where appropriate.

The objective is not simply to reproduce spreadsheet calculations. Each topic should be approached by understanding the underlying financial relationship, implementing the calculation in Python, validating the result, and interpreting what the result means financially.

---

## Learning Objectives

By the end of this chapter, I should be able to:

- Explain the relationship between present value and future value.
- Calculate present value (PV) and net present value (NPV) for a series of cash flows.
- Calculate and interpret the internal rate of return (IRR).
- Explain why some cash-flow patterns can produce multiple IRRs.
- Construct and analyze loan amortization tables.
- Develop flat-payment loan schedules.
- Calculate future value (FV) and apply it to financial planning problems.
- Model more complex future-value problems, including pension scenarios.
- Explain and calculate continuously compounded returns.
- Discount irregularly dated cash flows using actual cash-flow dates.
- Implement and validate these calculations using Python.

---

## Topics

### 1. Overview

Introduction to basic financial analysis and the time value of money.

### 2. Present Value and Net Present Value

- Present Value (PV)
- Discount factors
- Cash-flow timing
- Net Present Value (NPV)
- Investment decision rules

### 3. Internal Rate of Return and Loan Tables

- Internal Rate of Return (IRR)
- Relationship between NPV and IRR
- Loan balances
- Interest and principal payments
- Amortization tables

### 4. Multiple Internal Rates of Return

- Non-conventional cash flows
- Multiple IRR solutions
- Limitations of IRR
- NPV profiles and interpretation

### 5. Flat Payment Schedules

- Level-payment loans
- Periodic payments
- Principal reduction
- Interest allocation
- Remaining loan balance

### 6. Future Value and Applications

- Future Value (FV)
- Compound growth
- Periodic contributions
- Savings and investment applications

### 7. A Pension Problem — Complicating the Future Value Problem

- Multi-stage cash flows
- Contributions over time
- Accumulation periods
- Retirement and pension modeling
- Combining PV and FV relationships

### 8. Continuous Compounding

- Discrete versus continuous compounding
- Exponential growth
- Continuously compounded rates
- Converting between compounding conventions

### 9. Discounting Using Dated Cash Flows

- Irregular cash-flow timing
- Date-based discounting
- Fractional periods
- Comparison with regularly spaced cash flows

---

## Python Implementation Goals

Where appropriate, the chapter exercises will be implemented using Python rather than Excel.

Implementations should favor transparent financial logic over simply reproducing spreadsheet behavior.

Potential tools include:

- Python standard library
- NumPy
- pandas
- SciPy
- Matplotlib
- `datetime`

Each implementation should, where practical:

1. Define the financial problem.
2. Identify the governing equation or relationship.
3. Implement the calculation in Python.
4. Validate the result against the book's example.
5. Interpret the result financially.
6. Test important edge cases or assumptions.

---

## Reference

Benninga, Simon, and Tal Mofkadi. *Financial Modeling*.

This chapter follows the organization and concepts presented in the text while developing independent Python implementations for educational purposes.

## Present Value

$$PV = \frac{Cash Flow}{(1 + r)^t}$$

Where:
- PV = Present Value
- r = Discount rate
- t = Time period

In [16]:
# Import necessary libraries
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import datetime
import scipy.optimize as opt
import seaborn as sns
from scipy.optimize import newton

In [2]:
# Present Value Function
def present_value(cash_flow, discount_rate, time_period):
    """
    Calculate the present value of a future cash flow.

    Parameters:
    - cash_flow: The future cash flow amount
    - discount_rate: The discount rate (as a decimal)
    - time_period: The time period until the cash flow occurs

    Returns:
    - The present value of the future cash flow
    """
    pv = cash_flow / (1 + discount_rate) ** time_period
    return pv

pv = present_value(100, 0.04, 5)
print(f"Present Value: ${pv:.2f}")

Present Value: $82.19


## Present Value of a Series of Cash Flows

$$
PV = A\left[\frac{1-(1+r)^{-n}}{r}\right]
$$

Where:
- PV = Present Value of the series of cash flows
- A = Amount of each cash flow
- r = Discount rate
- t = Time period for each cash flow

In [3]:
# Present Value of Periodic Series Function
def present_value_series(cash_flows, discount_rate, time_periods):
    """
    Calculate the present value of a series of future cash flows.

    Parameters:
    - cash_flows: A list or array of future cash flow amounts
    - discount_rate: The discount rate (as a decimal)
    - time_periods: A list or array of time periods corresponding to each cash flow

    Returns:
    - The present value of the series of future cash flows
    """
    pv_series = cash_flows * ((1 - (1 + discount_rate) ** -time_periods) / discount_rate)
    return pv_series

In [4]:
# Example: Present Value of a Series of Cash Flows
pv_series = present_value_series(100, 0.04, 5)
print(f"Present Value of Series: ${pv_series:.2f}")

Present Value of Series: $445.18


## Present Value of a Series of Cash Flows using numpy

$$
PV = \sum_{t=1}^{n} \frac{Cash Flow_t}{(1 + r)^t}
$$

Where:
- PV = Present Value
- r = Discount rate
- t = Time period
- Cash Flow_t = Cash flow at time t

In [5]:
# Present Value of a Series of Cash Flows using numpy
def present_value_series_numpy(cash_flows, discount_rate):
    """
    Calculate the present value of a series of future cash flows using numpy.

    Parameters:
    - cash_flows: A list or array of future cash flow amounts
    - discount_rate: The discount rate (as a decimal)

    Returns:
    - The present value of the series of future cash flows
    """
    cash_flows = np.array(cash_flows)
    time_periods = np.arange(1, len(cash_flows) + 1)
    pv_series = np.sum(cash_flows / (1 + discount_rate) ** time_periods)
    return pv_series


In [6]:
# Example: Present Value of a Series of Cash Flows using numpy
cash_flows = [30, 40, 50, 60]
pv_series_numpy = present_value_series_numpy(cash_flows, 0.04)
print(f"Present Value of Series using numpy: ${pv_series_numpy:.2f}")

Present Value of Series using numpy: $161.57


## Net Present Value (NPV)

The net present value (NPV) of a series of cash flows is the sum of the present values of each cash flow, minus the initial investment or asset value. It is a key metric in capital budgeting and investment analysis, helping to determine whether an investment is worthwhile.
$$
NPV = \sum_{t=0}^{n} \frac{Cash Flow_t}{(1 + r)^t} - Asset Value
$$

Where:
- NPV = Net Present Value
- r = Discount rate
- t = Time period
- Cash Flow_t = Cash flow at time t
- Asset Value = Initial investment or asset value (as a negative cash flow)

In [7]:
# Net Present Value Function
def net_present_value(cash_flows, discount_rate, asset_value=0):
    """
    Calculate the net present value of a series of cash flows.

    Parameters:
    - cash_flows: A list or array of cash flow amounts
    - discount_rate: The discount rate (as a decimal)
    - asset_value: The initial investment or asset value (as a negative cash flow)

    Returns:
    - The net present value of the cash flows
    """
    npv = sum(cf / (1 + discount_rate) ** t for t, cf in enumerate(cash_flows))
    npv -= asset_value
    return npv

# Example: Net Present Value
cash_flows = [-100, 30, 40, 50, 60]
discount_rate = 0.04
asset_value = 100
npv = net_present_value(cash_flows, discount_rate, asset_value)
print(f"Net Present Value: ${npv:.2f}")

Net Present Value: $-38.43


## Present Value of Annuities

The present value of an annuity is the current value of a series of equal payments made at regular intervals over a specified period of time, discounted at a given interest rate. An annuity can be either an ordinary annuity (payments made at the end of each period) or an annuity due (payments made at the beginning of each period).

$$
PV_{annuity} = A \left[ \frac{1 - (1 + r)^{-n}}{r} \right]
$$

Where:
- \(PV_{annuity}\) = Present Value of the annuity
- \(A\) = Amount of each annuity payment
- \(r\) = Discount rate per period
- \(n\) = Total number of payments

In [8]:
# Present Value of Annuity Function
def present_value_annuity(payment, discount_rate, num_payments):
    """
    Calculate the present value of an annuity.

    Parameters:
    - payment: The amount of each annuity payment
    - discount_rate: The discount rate per period (as a decimal)
    - num_payments: The total number of payments

    Returns:
    - The present value of the annuity
    """
    pv_annuity = payment * (1 - (1 + discount_rate) ** -num_payments) / discount_rate
    return pv_annuity


In [9]:
# Example: Present Value of Annuity
payment = 100
discount_rate = 0.04
num_payments = 5
pv_annuity = present_value_annuity(payment, discount_rate, num_payments)
print(f"Present Value of Annuity: ${pv_annuity:.2f}")

Present Value of Annuity: $445.18


In [10]:
# Present Value of Perpetuity Function
def present_value_perpetuity(payment, discount_rate):
    """
    Calculate the present value of a perpetuity.

    Parameters:
    - payment: The amount of each perpetuity payment
    - discount_rate: The discount rate per period (as a decimal)

    Returns:
    - The present value of the perpetuity
    """
    pv_perpetuity = payment / discount_rate
    return pv_perpetuity


In [11]:
# Example: Present Value of Perpetuity
payment = 100
discount_rate = 0.04
pv_perpetuity = present_value_perpetuity(payment, discount_rate)
print(f"Present Value of Perpetuity: ${pv_perpetuity:.2f}")

Present Value of Perpetuity: $2500.00


In [12]:
# Value of Finite Growing Annuity Function

def present_value_finite_growing_annuity(payment, discount_rate, growth_rate, num_payments):
    """
    Calculate the present value of a finite growing annuity.

    Parameters:
    - payment: The amount of the first annuity payment
    - discount_rate: The discount rate per period (as a decimal)
    - growth_rate: The growth rate of the annuity payments (as a decimal)
    - num_payments: The total number of payments

    Returns:
    - The present value of the finite growing annuity
    """
    if discount_rate == growth_rate:
        pv_finite_growing_annuity = payment * num_payments / (1 + discount_rate)
    else:
        pv_finite_growing_annuity = payment * (1 - ((1 + growth_rate) / (1 + discount_rate)) ** num_payments) / (discount_rate - growth_rate)

    return pv_finite_growing_annuity

In [13]:
# Example: Present Value of Finite Growing Annuity
payment = 1000
discount_rate = 0.06
growth_rate = 0.03
num_payments = 5
pv_finite_growing_annuity = present_value_finite_growing_annuity(payment, discount_rate, growth_rate, num_payments)
print(f"Present Value of Finite Growing Annuity: ${pv_finite_growing_annuity:.2f}")

Present Value of Finite Growing Annuity: $4457.43


## The Gordon Formula

The Gordon formula, also known as the Gordon Growth Model or Dividend Discount Model (DDM), is a method for valuing a stock by assuming that dividends will grow at a constant rate indefinitely. It is particularly useful for valuing companies with stable dividend growth.

$$
P_0 = \frac{D_1}{r - g}
$$

Where:
- \(P_0\) = Current stock price
- \(D_1\) = Dividend expected in the next period
- \(r\) = Required rate of return (discount rate)
- \(g\) = Constant growth rate of dividends

In [14]:
# Gordon Formula Function
def gordon_formula(dividend_next_period, discount_rate, growth_rate):
    """
    Calculate the current stock price using the Gordon Growth Model.

    Parameters:
    - dividend_next_period: The dividend expected in the next period (D1)
    - discount_rate: The required rate of return (r)
    - growth_rate: The constant growth rate of dividends (g)

    Returns:
    - The current stock price (P0)
    """
    stock_price = dividend_next_period / (discount_rate - growth_rate)
    return stock_price

In [15]:
# Example: Gordon Formula
dividend_next_period = 2.00  # Expected dividend next period
discount_rate = 0.08  # Required rate of return
growth_rate = 0.04  # Constant growth rate of dividends
stock_price = gordon_formula(dividend_next_period, discount_rate, growth_rate)
print(f"Current Stock Price using Gordon Formula: ${stock_price:.2f}")

Current Stock Price using Gordon Formula: $50.00


## Internal Rate of Return (IRR) and Loan Tables

The internal rate of return (IRR) is the discount rate that makes the net present value (NPV) of a series of cash flows equal to zero. It is a key metric in capital budgeting and investment analysis, helping to determine the profitability of an investment.

$$
NPV = \sum_{t=0}^{n} \frac{C_t}{(1 + IRR)^t} = 0
$$

In [17]:
# Internal Rate of Return Function
def internal_rate_of_return(cash_flows):
    """
    Calculate the internal rate of return (IRR) for a series of cash flows.

    Parameters:
    - cash_flows: A list of cash flows (C0, C1, ..., Cn)

    Returns:
    - The internal rate of return (IRR)
    """

    # Define the NPV function
    def npv(irr):
        return sum(cf / (1 + irr) ** i for i, cf in enumerate(cash_flows))

    # Use the Newton-Raphson method to find the IRR
    irr = newton(npv, 0.1)  # Initial guess of 10%
    return irr

In [18]:
# Example: Internal Rate of Return
cash_flows = [-100, 30, 40, 50, 60]
irr = internal_rate_of_return(cash_flows)
print(f"Internal Rate of Return (IRR): {irr:.4%}")

Internal Rate of Return (IRR): 24.8883%
